# OOD Detection
The purpose of this lab project is to enhance our understanding of OOD detection. After accomplishing the lab project, you should be able to:
- Code different OOD score functions and use them for OOD detection.
- Perform benchmarking experiments involving different OOD score functions and different metrics.
- Visualize OOD detection results and check for common mistakes in OOD detection experiments.

As usual, we start by importing the necessary libraries.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, precision_recall_curve
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## 1. Data
The ultimate purpose of this notebook is to perform a benchmarking experiment in order to compare multiple OOD scores and OOD detection algorithms. To that end, we will use three different data sets:
1. The **Cifar-10 train** dataset in order to train a simple convolutional neural network for the task of image classification.
2. The **Cifar-10 test** set as the *in-distribution* dataset (i.e. the dataset of normal examples), for evaluating the different OOD scores.
3. (A subset of) The **SVHN test** set as the *out-of-distribution* dataset (i.e. the dataset of anomalous examples), for evaluating the different OOD scores.

**Question.** Would it be fair to use the *Cifar-10 train* set along with the SVHN test set in order to evaluate the OOD detection scores and algorithms?

In [ ]:
# Data loading and preprocessing
batch_size = 128

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

cifar_train = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
cifar_test = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)
svhn_test = datasets.SVHN(root='./data', split='test', transform=transform, download=True)

# Extract 10_000 random images from the svhn_test set
svhn_test, _ = torch.utils.data.random_split(svhn_test, [10_000, len(svhn_test) - 10_000])

train_loader = DataLoader(cifar_train, batch_size=batch_size, shuffle=True)
cifar_test_loader = DataLoader(cifar_test, batch_size=batch_size, shuffle=False)
svhn_test_loader = DataLoader(svhn_test, batch_size=batch_size, shuffle=False)

: 

In [ ]:
print(f"Number of training samples: {len(cifar_train)}")
print(f"Number of test samples: {len(cifar_test)}")
print(f"Number of SVHN test samples: {len(svhn_test)}")

Number of training samples: 50000
Number of test samples: 10000
Number of SVHN test samples: 10000


## 2. CNN Classifier
We will first train a CNN Classifier on the Cifar-10 training data, for the task of classifying the Cifar-10 images. This CNN will output the logit values. Bare in mind that some of the OOD scores we will define require access to the features of the penultimate layer.

**Exercise** Add a `return_features` argument to the `forward` method, defaulting to `False`. If `return_features` is set to `True`, the `forward` method should return the features of the penultimate layer instead of the logit values.

In [ ]:
# Define a simple CNN model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

    # TODO: Add an optional argument to return the features from the penultimate layer
    def forward(self, x, ...):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        features = self.relu(self.fc1(x))
        # TODO: return features if return_features is True
        logits = self.fc2(features)
        return logits

In [ ]:
# %load solutions/classifier.py

## 3. Training
We train the above CNN on the Cifar-10 training set.

In [ ]:
model = SimpleCNN().to(device)

# Hyper-parameters
num_epochs = 5
learning_rate = 0.001

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
def train_model():
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}")

train_model()

In [ ]:
# print test loss and accuracy
model.eval
with torch.no_grad():
    test_loss = 0
    correct = 0
    total = 0
    for images, labels in cifar_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Test Loss: {test_loss / len(cifar_test_loader):.4f}")
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

## 4. OOD Metrics
The objective of this section is to define the different OOD metrics studied during the lectures. Recall that we have seen two kinf of metrics:
1. Fixed-threshold metrics.
2. Threshold-independent metrics.

### 4.1. Fixed-threshold metrics
We will start to define the metrics for OOD detectors with a fixed threshold. The inputs to all of our metrics below will be:
- The `scores_negatives` nupy array: an array containing the scores for the ground truth negative images (i.e. the Cifar-10 test images).
- The `scores_positives` numpy array: an array containing the scores for the ground truth positive images (i.e. the SVHN test images).
- The `threshold` floating point number. The threshold value $\tau$ such that our OOD detector classifies examples according to their score as follwos:
$$\begin{cases}
s \leq \tau\quad &⇒\quad \text{negative}\\
s > \tau\quad &⇒\quad \text{positive}
\end{cases}$$
- Any other parameters necessary for the metric in question.

**Exercise.** Define the functions below:
1. A `confusion_matrix` function that outputs the number of *false positives*, *true positives*, *true negatives* and *false negatives*.
2. A `tpr_fpr` function that outputs the  *true positive rate* and *false positive rate*.
3. An `accuracy` function that outputs the accuracy.
4. A `precission_recall` function that outputs the *precision* and the *recall*.
5. An `f_beta` function that takes an additional input argument `beta` and returns the corresponding $F_\beta$ score.

In [ ]:
def confusion_matrix(scores_negatives, scores_positives, threshold):
    # TODO: Compute and return the confusion matrix

def tpr_fpr(scores_negatives, scores_positives, threshold):
    # TODO: Compute and return the tpr and fpr

def accuracy(scores_negatives, scores_positives, threshold):
    # TODO: Compute and return the accuracy

def precision_recall(scores_negatives, scores_positives, threshold):
    # TODO: Compute and return the precission and recall

def f_beta(scores_negatives, scores_positives, threshold, beta):
    # TODO: Compute and return the f_beta score

In [ ]:
# %load solutions/ftmetrics.py

### 4.2. Threshold-independent metrics
**Exercise.** Define the function `roc_auc` that:
- Takes as input the `scores_negatives` and `scores_positives` numpy arrays.
- Plots the *ROC curve*.
- Returns the value of the *AUROC* as the area under the *ROC curve*.

In [ ]:
def roc_auc(scores_negatives, scores_positives):
    # TODO: Combine scores and create labels
    scores = np.concatenate((scores_negatives, scores_positives))
    labels = ... # TODO Give the label 0 to negative data and the label 1 to positive data

    # Sort scores and labels
    sorted_indices = np.argsort(scores)
    scores = scores[sorted_indices]
    labels = labels[sorted_indices]

    # Initialize TPR and FPR
    tpr = []
    fpr = []
    n_pos = np.sum(labels)
    n_neg = len(labels) - n_pos

    tp = n_pos
    fp = n_neg

    # TODO: loop through all possible thresholds (i.e. all possible scores)
    # and update the number of true positives and false positives for each threshold.
    # Compute the respective tpr and fpr and append them to the tpr and fpr lists.

    # Convert the tpr and fpr lists to numpy arrays
    tpr = np.array(tpr)
    fpr = np.array(fpr)

    # Compute AUROC (Area Under the Curve)
    auroc = # TODO: Compute the AUC using the np.trapezoid function

    # Plot ROC curve
    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC Curve (AUROC = {auroc:.4f})")
    plt.plot([0, 1], [0, 1], 'k--', label="Random Classifier")
    plt.xlabel("False Positive Rate (FPR)")
    plt.ylabel("True Positive Rate (TPR)")
    plt.title("Receiver Operating Characteristic (ROC) Curve")
    plt.legend(loc="lower right")
    plt.grid()
    plt.show()

    return auroc

In [ ]:
# %load solutions/roc_auc.py


## 5. OOD Scores
In this section, we will implement the different OOD scores seen during the lecture. Recall that we can split the different OOD scores into two score families:
1. Logit-based scores.
2. Feature-based scores.

### 5.1. Logit-based scores
Logit-based scores are simpler to implement than feature-based scores. We will implement each of the logit-based scores as a function that takes as inputs the `logits` array of logits of the different test points,
and returns the array of test point scores.

**Exercise.** Complete the functions below with the formulas seen during the lecture.

In [ ]:
# MLS Score
def mls(logits):
    # TODO: Compute and return the MLS score

# MSP Score
def msp(logits):
    # TODO: Compute and return the MSP score

# Energy Score
def energy(logits, temp=1):
    # TODO: Compute and return the Energy score

# Entropy Score
def entropy(logits):
    # TODO: Compute and return the Entropy score

In [ ]:
# %load solutions/logit_scores.py

### 5.2. DKNN
In this section we define a class `DKNN` to compute the Deep $K$-nearest neighbor score. This score is more involved than the previous ones for two main reasons:
- It employs the activations of the penultimate layer of the CNN rather than the logit or softmax values.
- It requires a fitting dataset in order to compute distances of the test images with respect to the images in the fitting dataset. We will be using the Cifar-10 training set as fitting dataset.

*Exercise.* Complete the following methods in the class `DKNN` below:
1. The `_l2_normalization` method that normalizes a batch of feature vectors by dividing each feature vector by its $\ell_2$ norm.
2. The `compute_scores` function that computes the distance from each of the test points to its $k$-th nearest neighbor in the fit dataset. The distances are computed between the normalized feature representations. The test points are processed in batches to avoid memory issues.

In [ ]:
class DKNN:
    def __init__(self, k=50, batch_size=256):
        self.k = k
        self.batch_size = batch_size
        self.fit_features = None

    def _l2_normalization(self, feat):
        norms = ... # TODO: Compute the norm of each feature vector, and add a small constant to it to avoid dividing by zero
        return feat / norms

    def fit(self, fit_dataset):
        self.fit_features = # TODO: Apply the l2 normalization to the fit dataset.

    def compute_scores(self, test_features):
        test_features = ... # TODO: Apply the l2 normalization to the test dataset.
        scores = []

        # Process test features in batches
        for i in range(0, test_features.size(0), self.batch_size):
            batch = test_features[i:i + self.batch_size]
            # Compute pairwise distances for the batch
            distances = torch.cdist(batch, self.fit_features, p=2)  # (batch_size, num_fit_samples)
            # TODO: Sort distances and extract the k-th nearest
            # Append the results to the list of scores.


        # Concatenate scores from all batches
        return torch.cat(scores, dim=0).cpu().numpy()

In [ ]:
# %load solutions/dknn.py

### 5.3. Mahalanobis
In this section we define a class `Mahalanobis` to compute the Mahalanobis score. This class is similar to the `DKNN` for the same reasons as before:
- It employs the activations of the penultimate layer of the CNN rather than the logit or softmax values.
- It requires a fitting dataset in order to compute distances of the test images with respect to the images in the fitting dataset. We will be using the Cifar-10 training set as fitting dataset.

*Exercise.* Complete the following methods in the class `Mahalanobis` below:
1. The `fit` method that fits per-class mean vectors and a common covariance matrix to the fitting dataset.
2. The `_mahalanobis_distance` method that computes the Mahalanobis distance of a given vector with respect to the gaussian law parametrized by its mean vector and covariance matrix.
3. The `compute_scores` function that uses the two previous methods to compute the Mahalanobis score of all test points by taking the maximum of Mahalanobis distances over the set of different classes/labels.

In [ ]:
class Mahalanobis():
    def __init__(self):
        self.mus = None
        self.inv_cov = None
        self.labels = None

    def fit(self, features, labels):
        self.labels = # TODO: extract the set of unique labels
        self.mus = {}
        covs = {}
        for label in self.labels:
            # TODO: fit the mean vector corresponding to the label
            # and the RESCALED covariance matrix

        cov = # TODO: Compute the common covariance matrix for all labels
        self.inv_cov = # TODO: Compute the (pseudo-)inverse of the covariance matrix

    def _mahalanobis_distance(self, x, mu, inv_cov):
        # TODO: Compute and return the Mahalanobis distance for the given mean and inverse covariance

    def compute_scores(self, test_features):
        scores = []
        for test_feature in test_features:
            distances = # TODO: Compute the vectore of per-label Mahalanobis distances
            # TODO: Compute the mahalanobis score of the current test example and append it to the list of scores.
        return torch.stack(scores).cpu().numpy()

In [ ]:
# %load solutions/mahalanobis.py

## 6. Score Comparison
The objective of this section is to compare the different OOD scores that we have just defined. Note that in order to use the *threshold-dependent metrics*, we need to pick a threshold for each of the scores.

Picking the same threshold for all scores *is not* a proper way to compare the different scores, since thy are scaled differently. A common way to perform a more "fair" comparison is to do the following:
1. Fix a target TPR, e.g. 0.95.
2. Compute the threshold $\tau$ such that the TPR on the SVHN test dataset is equal to the target TPR 0.95.
3. Compute the remaining fixed-threshold metrics for such $\tau$.

**Exercise.** Define the function `compute_threshold` that:
- Takes as inputs `scores`, a numpy array of scores and a `target_tpr`, a value between 0 and 1 defaulting to 0.95.
- Assuming that the array of `scores` contains the scores of the positive examples, the function computes and returns the value of the threshold $\tau$ that achieves the desired `target_tpr`.

In [ ]:
def compute_threshold(scores, target_tpr=0.95):
    # TODO: Compute and return the desired threshold.

In [ ]:
# %load solutions/compute_threshold.py

In order to compare the different OOD scores that we have defined, we set the variable `target_tpr` equal to 0.9 and we initialise an empty dictionary to store the different metrics for the different OOD scores.

In [ ]:
target_tpr = 0.9
metrics_dict = {}

### 6.1. Metrics for logit-based scores

Next we compute the different evaluation metrics for each of the scores above, starting with the *logit-based scores*. We start by extracting the logits of the Cifar-10 test set and the SVHN test set.


In [ ]:
# Compute logits directly from the dataset
def compute_logits(dataset, model, device):
    all_logits = []
    with torch.no_grad():
        for i in range(len(dataset)):
            image, _ = dataset[i]  # Get each image directly from the dataset
            image = image.unsqueeze(0).to(device)  # Add batch dimension and move to device
            logits = model(image)
            all_logits.append(logits)
    return torch.cat(all_logits, dim=0)  # Concatenate all logits into a single tensor

# Apply the function to CIFAR-10 and SVHN datasets
test_logits_negatives = compute_logits(cifar_test, model, device)
test_logits_positives = compute_logits(svhn_test, model, device)

**Exercise.** For each of the *MLS*, *MSP*, *Energy (T=1)* and *Entropy* OOD score functions:
    - Compute the scores on the Cifar-10 test set and the SVHN test set.
    - Plot the histogram of the scores and check that the negative samples have, on average, lower scores than the positive samples.
    - Use the `roc_auc` function to plot the ROC curves and compute the AUROCs.
    - Compute the trhreshold that achieves 0.1 FPR and compute the fixed-threshold metrics associated to it: accuracy, TPR, Precision, Recall and $F_1$.
    - Store all the metrics in the `metrics_dict` dictionary for future comparison.

In [ ]:
scoring_functions = {
    'MLS': mls,
    'MSP': msp,
    'Energy': energy,
    'Entropy': entropy
}

for method, scoring_function in scoring_functions.items():

    # TODO: Compute scores
    scores_negatives = ...
    scores_positives = ...

    # Plot histogram of scores
    plt.figure(figsize=(10, 6))
    plt.hist(scores_negatives, bins=50, alpha=0.5, label='Negative Samples')
    plt.hist(scores_positives, bins=50, alpha=0.5, label='Positive Samples')
    plt.xlabel('Score')
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {method} Scores')
    plt.legend()
    plt.show()
    
    # Initialize empty dict for metrics
    metrics_dict[method] = {}

    # TODO: Plot ROC curve and compute AUROC
    auroc = ...
    metrics_dict[method]['auroc'] = auroc

    # TODO: Compute threshold for the given target_tpr
    threshold = ...

    # TODO: Compute and store remaining metrics

In [ ]:
# %load solutions/logits_comparison.py

### 6.2. Metrics for feature-based scores

We extract the representations in the feature space given by the penultimate layer of the CNN of the three datasets: Cifar-10 training dataset, Cifar-10 test set and SVHN test set.

In [ ]:
# Compute features directly from the dataset
def compute_features(dataset, model, device):
    all_features = []
    with torch.no_grad():
        for i in range(len(dataset)):
            image, _ = dataset[i]  # Get each image directly from the dataset
            image = image.unsqueeze(0).to(device)  # Add batch dimension and move to device
            features = model(image, return_features=True)
            all_features.append(features)
    return torch.cat(all_features, dim=0)  # Concatenate all logits into a single tensor

# Apply the function to CIFAR-10 train, test, and SVHN test datasets
fit_features = compute_features(cifar_train, model, device)
test_features_negatives = compute_features(cifar_test, model, device)
test_features_positives = compute_features(svhn_test, model, device)

**Exercise.**
1. Compute the *DKNN scores* for the Cifar-10 test dataset and the SVHN test datsets using the 5-th nearest neighbor.
2. Plot the histogram of the scores and check that the negative samples have, on average, lower scores than the positive samples.
2. Use the `roc_auc` function to plot the ROC curve and compute the AUROC.
3. Compute the trhreshold that achieves 0.1 FPR and compute the fixed-threshold metrics associated to it: accuracy, TPR, Precision, Recall and $F_1$.
4. Store all the metrics in the `metrics_dict` dictionary for future comparison.


In [ ]:
metrics_dict['DKNN'] = {}

# TODO: initialize and fit the DKNN model

# TODO: Compute the scores of the negative and positive data from their feature representations

# Plot the histogram of the scores
plt.figure(figsize=(10, 6))
plt.hist(scores_negatives, bins=50, alpha=0.5, label='Negative Samples')
plt.hist(scores_positives, bins=50, alpha=0.5, label='Positive Samples')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title(f'Histogram of {method} Scores')
plt.legend()
plt.show()

# TODO: Plot ROC curve and compute AUROC
auroc = ...
metrics_dict['DKNN']['auroc'] = auroc

# TODO: Compute threshold for the given target_tpr
threshold = ...

# TODO: Compute and store remaining metrics

In [ ]:
# %load solutions/dknn_metrics.py

**Exercise.**
1. Compute the *Mahalanobis scores* for the Cifar-10 test dataset and the SVHN test datsets using the 5-th nearest neighbor.
2. Plot the histogram of the scores and check that the negative samples have, on average, lower scores than the positive samples.
2. Use the `roc_auc` function to plot the ROC curve and compute the AUROC.
3. Compute the trhreshold that achieves 0.1 FPR and compute the fixed-threshold metrics associated to it: accuracy, TPR, Precision, Recall and $F_1$.
4. Store all the metrics in the `metrics_dict` dictionary for future comparison.

In [ ]:
metrics_dict['Mahalanobis'] = {}

# TODO: initialize and fit the Mahalanobis model

# TODO: Compute the scores of the negative and positive data from their feature representations

# Plot the histogram of the scores
plt.figure(figsize=(10, 6))
plt.hist(scores_negatives, bins=50, alpha=0.5, label='Negative Samples')
plt.hist(scores_positives, bins=50, alpha=0.5, label='Positive Samples')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title(f'Histogram of {method} Scores')
plt.legend()
plt.show()

# TODO: Plot ROC curve and compute AUROC
auroc = ...
metrics_dict['Mahalanobis']['auroc'] = auroc

# TODO: Compute threshold for the given target_tpr
threshold = ...

# TODO: Compute and store remaining metrics

In [ ]:
# %load solutions/mahalanobis_metrics.py

## Results Table
**Exercise.** Plot the results stored in the dictionary `metrics_dict` by highlighting the method that achieves the best value for each of the different metrics.

In [ ]:
import pandas as pd

# TODO: Display a table with the best results highlited.
# Careful! The best result is not always the maximum value!

In [ ]:
# %load solutions/results_table.py

**Bonus Exercises.** If you still have time at the end, you can try and do the following:
1. Play with different temperature parameters in the *Energy* score to see how they affect the different metrics.
2. Play with different $k$ parameters in the *DKNN* algorithm to see how they affect the different metrics.
3. Write docstirngs for the above function (In the future, you will be greatful to your current self if you find yourself checking out this notebook and the docstrings are there).
4. Download a better model (e.g. a pre-trained VGG model fine-tuned on Cifar-10) and check out if you get better results with it.
5. Check out the OODEEL library where a benchmark like the one we have just carried-out is much easier to perform ;)

## 7. OOD Detection in 2D Feature Space with MNIST

In the previous sections, we defined and benchmarked OOD scores on a realistic high-dimensional setting (CIFAR-10 vs SVHN). However, because the feature representations are 128-dimensional, it is difficult to visualise *geometrically* how each score behaves.

In this section, we train a simpler network on a simpler dataset (**MNIST**) with the constraint that the **penultimate layer is 2-dimensional**. This gives us:
1. A feature space we can draw on paper.
2. A direct view of the **decision landscape** (score contours) for each OOD detector.

**Setup**
- **In-distribution (ID)**: MNIST digits 0, 1, 2, 3, 4.
- **Out-of-distribution (OOD)**: MNIST digits 5, 6, 7, 8, 9.

**Convention (same as before)**: higher score ⇒ more likely to be OOD.

### 7.1 Data

We load MNIST and split it into:
- **`mnist_train_id`**: MNIST training images with labels 0–4 (used for fitting OOD scores).
- **`mnist_test_id`**: MNIST test images with labels 0–4 (ID test set).
- **`mnist_test_ood`**: MNIST test images with labels 5–9 (OOD test set).

In [ ]:
# Load full MNIST datasets
mnist_transform = transforms.Compose([transforms.ToTensor()])

mnist_train_full = datasets.MNIST(root='./data', train=True, transform=mnist_transform, download=True)
mnist_test_full = datasets.MNIST(root='./data', train=False, transform=mnist_transform, download=True)

# Split into ID (classes 0-4) and OOD (classes 5-9)
in_labels = [0, 1, 2, 3, 4]

mnist_train_id_indices = [i for i, (_, label) in enumerate(mnist_train_full) if label in in_labels]
mnist_test_id_indices = [i for i, (_, label) in enumerate(mnist_test_full) if label in in_labels]
mnist_test_ood_indices = [i for i, (_, label) in enumerate(mnist_test_full) if label not in in_labels]

mnist_train_id = torch.utils.data.Subset(mnist_train_full, mnist_train_id_indices)
mnist_test_id = torch.utils.data.Subset(mnist_test_full, mnist_test_id_indices)
mnist_test_ood = torch.utils.data.Subset(mnist_test_full, mnist_test_ood_indices)

# Create DataLoaders
mnist_batch_size = 256
mnist_train_id_loader = DataLoader(mnist_train_id, batch_size=mnist_batch_size, shuffle=True)
mnist_test_id_loader = DataLoader(mnist_test_id, batch_size=mnist_batch_size, shuffle=False)
mnist_test_ood_loader = DataLoader(mnist_test_ood, batch_size=mnist_batch_size, shuffle=False)

print(f"MNIST ID train samples: {len(mnist_train_id)}")
print(f"MNIST ID test samples:  {len(mnist_test_id)}")
print(f"MNIST OOD test samples: {len(mnist_test_ood)}")

In [ ]:
# Visualise some ID and OOD samples
def show_mnist_samples(loader, title, n=8):
    images, labels = next(iter(loader))
    images, labels = images[:n], labels[:n]
    fig, axes = plt.subplots(1, n, figsize=(1.5 * n, 1.5))
    for i in range(n):
        axes[i].imshow(images[i, 0].numpy(), cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(int(labels[i]))
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_mnist_samples(mnist_test_id_loader, "ID samples (MNIST digits 0–4)")
show_mnist_samples(mnist_test_ood_loader, "OOD samples (MNIST digits 5–9)")

: 

### 7.2 A CNN with a 2D Penultimate Layer

We define a simple convolutional network (`ToyConvnet2D`) whose **penultimate layer has only 2 neurons**. This 2D bottleneck forces the network to encode each image as a single 2D point, which we can directly plot.

Architecture:
- Conv2d(1 → 32, kernel 3) → ReLU → MaxPool(2)
- Conv2d(32 → 64, kernel 3) → ReLU → MaxPool(2)
- Flatten
- **Linear(64 × 5 × 5 → 2)** — the 2D penultimate layer
- Linear(2 → 10) — logits for 10 classes

The network is trained on classes 0–4 only, so classes 5–9 are never seen during training and will serve as OOD at test time.

**Exercise.** Implement `ToyConvnet2D`:
- Add a `return_features` argument to `forward` (default `False`). When `True`, return the 2D penultimate activations instead of the logits.

In [ ]:
class ToyConvnet2D(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 5 * 5, 2)   # 2D penultimate layer
        self.fc2 = nn.Linear(2, num_classes)    # output logits

    def forward(self, x, return_features=False):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        features = self.fc1(x)
        if return_features:
            return features
        return self.fc2(features)

mnist_model = ToyConvnet2D(num_classes=10).to(device)
print(mnist_model)

: 

### 7.3 Training on MNIST[0–4]

The model is trained as a standard 10-class classifier using cross-entropy loss, but it only sees digits 0–4 during training. Digits 5–9 are withheld and will be used as OOD samples.

In [ ]:
# Hyper-parameters
mnist_num_epochs = 10
mnist_lr = 1e-3

# Loss and optimizer
mnist_criterion = nn.CrossEntropyLoss()
mnist_optimizer = optim.Adam(mnist_model.parameters(), lr=mnist_lr)

# Training loop
mnist_model.train()
for epoch in range(mnist_num_epochs):
    running_loss = 0.0
    for images, labels in mnist_train_id_loader:
        images, labels = images.to(device), labels.to(device)
        mnist_optimizer.zero_grad()
        outputs = mnist_model(images)
        loss = mnist_criterion(outputs, labels)
        loss.backward()
        mnist_optimizer.step()
        running_loss += loss.item() * images.size(0)
    avg_loss = running_loss / len(mnist_train_id)
    print(f"Epoch [{epoch + 1}/{mnist_num_epochs}], Loss: {avg_loss:.4f}")


: 

### 7.4 Visualising the 2D Feature Space

We implement the `compute_mnist_features(dataset, model, device)` that returns the 2D penultimate features and the corresponding labels for all samples in `dataset`. Then:
1. We extract features for `mnist_train_id`, `mnist_test_id`, and `mnist_test_ood`.
2. We plot the training features coloured by class label (classes 0–4).
3. We plot the test features coloured by ID vs OOD.

In [ ]:
def compute_mnist_features(dataset, model, device):
    model.eval()
    all_features = []
    all_labels = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            features = model(images, return_features=True)
            all_features.append(features.cpu())
            all_labels.append(labels)
    return torch.cat(all_features, dim=0), torch.cat(all_labels, dim=0)

# Extract features for the three datasets
mnist_fit_features, mnist_fit_labels = compute_mnist_features(mnist_train_id, mnist_model, device)
mnist_id_features, mnist_id_labels = compute_mnist_features(mnist_test_id, mnist_model, device)
mnist_ood_features, mnist_ood_labels = compute_mnist_features(mnist_test_ood, mnist_model, device)

print("Feature shapes (2D penultimate):")
print(f"  mnist_fit_features : {mnist_fit_features.shape}")
print(f"  mnist_id_features  : {mnist_id_features.shape}")
print(f"  mnist_ood_features : {mnist_ood_features.shape}")

# Plot 2D features of the training set, colored by class label
colors = plt.cm.tab10(np.linspace(0, 0.5, 5))
plt.figure(figsize=(8, 7))
for i, c in enumerate(in_labels):
    idx = mnist_fit_labels == c
    plt.scatter(mnist_fit_features[idx, 0], mnist_fit_features[idx, 1],
                s=5, alpha=0.4, label=f"Class {c}", color=colors[i])
plt.title("2D penultimate features on MNIST[0-4] (train)")
plt.xlabel(r"$Z_1$")
plt.ylabel(r"$Z_2$")
plt.legend(markerscale=3, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Plot ID vs OOD test features
plt.figure(figsize=(8, 7))
plt.scatter(mnist_id_features[:, 0], mnist_id_features[:, 1],
            s=5, alpha=0.5, label="ID (0-4)")
plt.scatter(mnist_ood_features[:, 0], mnist_ood_features[:, 1],
            s=5, alpha=0.5, label="OOD (5-9)")
plt.title("2D penultimate features on MNIST test")
plt.xlabel(r"$Z_1$")
plt.ylabel(r"$Z_2$")
plt.legend()
plt.tight_layout()
plt.show()


### 7.5 OOD Score Landscapes

We will now visualise **how each OOD score partitions the 2D feature space**. For each method:
1. Create a fine 2D grid covering the entire feature space.
2. Compute the OOD score at every grid point.
3. Display the score as a colour map (`plt.contourf`), then overlay the ID and OOD test points.

This reveals the **decision boundary** of each OOD detector: the set of feature-space points where the score crosses the detection threshold.

In [ ]:
# Build a 2D grid covering the feature space
all_features_np = torch.cat([mnist_fit_features, mnist_id_features, mnist_ood_features], dim=0).numpy()

margin = 0.5
x_min = all_features_np[:, 0].min() - margin
x_max = all_features_np[:, 0].max() + margin
y_min = all_features_np[:, 1].min() - margin
y_max = all_features_np[:, 1].max() + margin

grid_size = 150
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, grid_size),
    np.linspace(y_min, y_max, grid_size),
)
grid_points = np.stack([xx.ravel(), yy.ravel()], axis=-1)  # (grid_size^2, 2)

print(f"Grid range: x in [{x_min:.2f}, {x_max:.2f}], y in [{y_min:.2f}, {y_max:.2f}]")
print(f"Grid shape: {grid_points.shape}")


#### 7.5.1 MLS Score Landscape

Recall that the MLS score is $s_{\text{MLS}}(x) = -\max_c f_c(x)$, where $f_c(x)$ are the logits. Since the last layer of `mnist_model` is a linear map $z \mapsto Wz + b$, the logits are **linear** in the 2D features $z$. This means the MLS landscape has a piecewise-linear structure.

**Exercise.** Plot the MLS OOD score landscape:
1. Extract the weight matrix $W$ and bias $b$ of `mnist_model.fc2`.
2. Compute the logits for every grid point: `logits_grid = grid_points @ W.T + b`.
3. Apply `mls()` to get the scores and reshape to the grid.
4. Plot with `plt.contourf`, then overlay the ID and OOD test points.

In [ ]:
# TODO: Extract W (shape: num_classes × 2) and b (shape: num_classes) from mnist_model.fc2
W = ...
b = ...

# TODO: Compute logits for each grid point and apply the mls() score function
# logits_grid = grid_points @ W.T + b   (shape: grid_size^2 × num_classes)
logits_grid = ...
mls_grid = ...  # apply mls(), then reshape to (grid_size, grid_size)

# Plot MLS score landscape
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("MLS OOD score landscape in 2D feature space")

cs = axes[0].contourf(xx, yy, mls_grid, levels=30, cmap="RdYlGn_r")
axes[0].scatter(mnist_id_features[:, 0], mnist_id_features[:, 1],
                s=8, c="white", edgecolor="tab:blue", alpha=0.6, label="ID (0-4)")
axes[0].scatter(mnist_ood_features[:, 0], mnist_ood_features[:, 1],
                s=8, c="white", edgecolor="tab:red", alpha=0.6, label="OOD (5-9)")
axes[0].set_xlabel(r"$Z_1$")
axes[0].set_ylabel(r"$Z_2$")
axes[0].set_title("Score contours + test points")
axes[0].legend(loc="upper right")

cs2 = axes[1].contourf(xx, yy, mls_grid, levels=30, cmap="RdYlGn_r")
axes[1].set_xlabel(r"$Z_1$")
axes[1].set_ylabel(r"$Z_2$")
axes[1].set_title("Score contours only")
plt.colorbar(cs2, ax=axes[1], label="MLS OOD score (higher = more OOD)")
plt.tight_layout()
plt.show()

In [ ]:
# %load solutions/mnist_mls_boundary.py

#### 7.5.2 Mahalanobis Score Landscape

The Mahalanobis detector models each ID class as a Gaussian with class-specific mean and shared covariance matrix. The OOD score is the minimum squared Mahalanobis distance to all class means — large distance ⇒ more OOD.

**Exercise.** Plot the Mahalanobis OOD score landscape:
1. Initialise and fit a `Mahalanobis` object on `mnist_fit_features` and `mnist_fit_labels`.
2. Compute scores for every grid point using `compute_scores(torch.tensor(grid_points, dtype=torch.float32))`.
3. Plot the landscape and overlay the test points.

*Note.* Recall that the `Mahalanobis` class implemented in Section 5.3 returns **`-min_distance`** (lower = more OOD). To keep a consistent visual convention where *green = ID* and *red = OOD*, negate the scores before plotting.

In [ ]:
# TODO: Initialize and fit the Mahalanobis scorer on MNIST ID training features
maha_mnist = Mahalanobis()
# maha_mnist.fit(...)

# TODO: Compute Mahalanobis scores on the grid
# Hint: maha_mnist.compute_scores(torch.tensor(grid_points, dtype=torch.float32))
maha_grid = ...  # raw scores (lower = more OOD due to sign convention)

# TODO: Negate the scores so that higher = more OOD, and reshape to (grid_size, grid_size)
maha_grid_display = ...

# Plot Mahalanobis score landscape
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Mahalanobis OOD score landscape in 2D feature space")

cs = axes[0].contourf(xx, yy, maha_grid_display, levels=30, cmap="RdYlGn_r")
axes[0].scatter(mnist_id_features[:, 0], mnist_id_features[:, 1],
                s=8, c="white", edgecolor="tab:blue", alpha=0.6, label="ID (0-4)")
axes[0].scatter(mnist_ood_features[:, 0], mnist_ood_features[:, 1],
                s=8, c="white", edgecolor="tab:red", alpha=0.6, label="OOD (5-9)")
axes[0].set_xlabel(r"$Z_1$")
axes[0].set_ylabel(r"$Z_2$")
axes[0].set_title("Score contours + test points")
axes[0].legend(loc="upper right")

cs2 = axes[1].contourf(xx, yy, maha_grid_display, levels=30, cmap="RdYlGn_r")
axes[1].set_xlabel(r"$Z_1$")
axes[1].set_ylabel(r"$Z_2$")
axes[1].set_title("Score contours only")
plt.colorbar(cs2, ax=axes[1], label="Mahalanobis OOD score (higher = more OOD)")
plt.tight_layout()
plt.show()

In [ ]:
# %load solutions/mnist_maha_boundary.py

#### 7.5.3 DKNN Score Landscape

The DKNN detector computes, for each test point, the distance to its $k$-th nearest neighbour among the ID training features (after L2 normalisation). Points far from the ID training cloud receive a high score.

**Exercise.** Plot the DKNN OOD score landscape:
1. Initialise and fit a `DKNN` object (use `k=50`) on `mnist_fit_features`.
2. Compute scores for every grid point using `compute_scores(torch.tensor(grid_points, dtype=torch.float32))`.
3. Plot the landscape and overlay the test points.

*Note.* For DKNN, higher scores naturally indicate more OOD-like samples (no sign flip needed).

In [ ]:
# TODO: Initialize and fit the DKNN scorer on MNIST ID training features (k=50)
dknn_mnist = DKNN(k=50)
# dknn_mnist.fit(...)

# TODO: Compute DKNN scores on the grid (higher = more OOD)
dknn_grid = ...  # reshape to (grid_size, grid_size)

# Plot DKNN score landscape
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("DKNN OOD score landscape in 2D feature space")

cs = axes[0].contourf(xx, yy, dknn_grid, levels=30, cmap="RdYlGn_r")
axes[0].scatter(mnist_id_features[:, 0], mnist_id_features[:, 1],
                s=8, c="white", edgecolor="tab:blue", alpha=0.6, label="ID (0-4)")
axes[0].scatter(mnist_ood_features[:, 0], mnist_ood_features[:, 1],
                s=8, c="white", edgecolor="tab:red", alpha=0.6, label="OOD (5-9)")
axes[0].set_xlabel(r"$Z_1$")
axes[0].set_ylabel(r"$Z_2$")
axes[0].set_title("Score contours + test points")
axes[0].legend(loc="upper right")

cs2 = axes[1].contourf(xx, yy, dknn_grid, levels=30, cmap="RdYlGn_r")
axes[1].set_xlabel(r"$Z_1$")
axes[1].set_ylabel(r"$Z_2$")
axes[1].set_title("Score contours only")
plt.colorbar(cs2, ax=axes[1], label="DKNN OOD score (higher = more OOD)")
plt.tight_layout()
plt.show()

In [ ]:
# %load solutions/mnist_dknn_boundary.py

### 7.6 Summary

The three 2D visualisations illustrate the key **geometric** differences between the OOD detectors:

| Detector | Decision-boundary geometry | Score grows... |
|---|---|---|
| **MLS** | Piecewise-linear (polytope boundaries) | Away from the max-logit half-spaces |
| **Mahalanobis** | Ellipsoidal contours around class means | With Mahalanobis distance to nearest cluster |
| **DKNN** | Follows the density of the ID training cloud | With distance to the $k$-th nearest ID neighbour |

**Questions to reflect on:**
1. Which detector has the tightest ID region? Does that help or hurt when the OOD data overlaps with ID?
2. MLS is a global linear score — can you see any region of the 2D space where it would fail even though Mahalanobis or DKNN succeed?
3. The DKNN score depends on the density of training data. How does this relate to the choice of $k$?

**Warning:** Latent spaces in real applications are of much higher dimensions. One should not assume that intuition built on these 2D visualizations is still valid in higher dimension, in particular the l2 normalization in DKNN will have a very different impact !